# 07_deployment.ipynb

This notebook deploys the approved model package from SageMaker Model Registry to a SageMaker real-time endpoint.

The workflow creates a SageMaker Model from the registered model package, deploys an endpoint, invokes the endpoint with sample test data, and optionally deletes the endpoint to avoid ongoing costs.

## Import Project Modules

The project source code is organized in the `src/` directory at the repository root.

Because this notebook is executed from the `notebooks/` directory, the project root is added to the Python path so that modules from the `src` package can be imported.

In [1]:
import sys

sys.path.append("..")

## Import Dependencies

Import the required Python libraries and AWS SDK clients.

This notebook uses `boto3` directly instead of the SageMaker Python SDK.

In [2]:
from datetime import datetime, timezone
from pathlib import Path
import json

import boto3
import pandas as pd
from botocore.exceptions import ClientError

from src.config import (
    AWS_REGION,
    BUCKET_NAME,
    MODEL_REGISTRY_METADATA_KEY,
    TEST_DATA_KEY,
    TARGET_COLUMN,
    DEPLOYMENT_MODEL_NAME_PREFIX,
    DEPLOYMENT_ENDPOINT_CONFIG_NAME_PREFIX,
    DEPLOYMENT_ENDPOINT_NAME,
    INFERENCE_INSTANCE_TYPE,
)

## Initialize AWS Clients

Initialize the AWS clients required for Amazon S3, SageMaker, SageMaker Runtime, IAM, and STS.

In [3]:
s3 = boto3.client("s3", region_name=AWS_REGION)
sm = boto3.client("sagemaker", region_name=AWS_REGION)
sm_runtime = boto3.client("sagemaker-runtime", region_name=AWS_REGION)
iam = boto3.client("iam", region_name=AWS_REGION)
sts = boto3.client("sts", region_name=AWS_REGION)

## Resolve SageMaker Execution Role

Resolve the SageMaker execution role from the current Studio execution context.

The IAM role ARN is loaded through IAM to preserve the full role path, such as `service-role`.

In [7]:
identity = sts.get_caller_identity()

caller_arn = identity["Arn"]
role_name = caller_arn.split(":assumed-role/")[1].split("/")[0]

role_response = iam.get_role(
    RoleName=role_name,
)

execution_role_arn = role_response["Role"]["Arn"]

print(f"SageMaker execution role: {execution_role_arn}")

SageMaker execution role: arn:aws:iam::591874026136:role/service-role/AmazonSageMaker-ExecutionRole-20260615T111026


## Define Deployment Configuration

Define SageMaker resource names for the deployment.

The SageMaker Model and endpoint configuration use timestamped names, while the endpoint uses a stable name.

In [8]:
timestamp = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

DEPLOYMENT_MODEL_NAME = f"{DEPLOYMENT_MODEL_NAME_PREFIX}-{timestamp}"
ENDPOINT_CONFIG_NAME = f"{DEPLOYMENT_ENDPOINT_CONFIG_NAME_PREFIX}-{timestamp}"

print(f"Deployment model name: {DEPLOYMENT_MODEL_NAME}")
print(f"Endpoint config name: {ENDPOINT_CONFIG_NAME}")
print(f"Endpoint name: {DEPLOYMENT_ENDPOINT_NAME}")
print(f"Instance type: {INFERENCE_INSTANCE_TYPE}")

Deployment model name: german-credit-risk-deployment-model-20260628175413
Endpoint config name: german-credit-risk-endpoint-config-20260628175413
Endpoint name: german-credit-risk-endpoint
Instance type: ml.m5.large


## Load Registry Metadata

Load the registry metadata created in the model registry notebook.

The metadata contains the registered model package ARN that will be deployed.

In [9]:
registry_metadata_obj = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=MODEL_REGISTRY_METADATA_KEY,
)

registry_metadata = json.load(registry_metadata_obj["Body"])

print(json.dumps(registry_metadata, indent=2))

{
  "model_package_group_name": "german-credit-risk-xgboost",
  "model_package_arn": "arn:aws:sagemaker:eu-central-1:591874026136:model-package/german-credit-risk-xgboost/1",
  "model_package_version": 1,
  "model_package_status": "Completed",
  "model_approval_status": "PendingManualApproval",
  "model_data_s3_uri": "s3://aws-sagemaker-showcase/models/sagemaker/xgboost/model.tar.gz",
  "xgboost_image_uri": "492215442770.dkr.ecr.eu-central-1.amazonaws.com/sagemaker-xgboost:1.7-1",
  "evaluation_report_s3_uri": "s3://aws-sagemaker-showcase/evaluation/evaluation.json",
  "clarify_analysis_s3_uri": "s3://aws-sagemaker-showcase/clarify/output/german-credit-clarify-20260628154331/analysis.json",
  "clarify_job_name": "german-credit-clarify-20260628154331",
  "sagemaker_model_name": "german-credit-xgboost-model-20260628152155",
  "created_at": "20260628162244"
}


In [10]:
model_package_arn = registry_metadata["model_package_arn"]

print(f"Model package ARN: {model_package_arn}")

Model package ARN: arn:aws:sagemaker:eu-central-1:591874026136:model-package/german-credit-risk-xgboost/1


## Verify Approved Model Package

Verify that the registered model package exists and is approved for deployment.

Only approved model packages should be deployed in this workflow.

In [11]:
model_package_description = sm.describe_model_package(
    ModelPackageName=model_package_arn,
)

approval_status = model_package_description["ModelApprovalStatus"]
model_package_status = model_package_description["ModelPackageStatus"]

print(f"Model package status: {model_package_status}")
print(f"Approval status: {approval_status}")

if approval_status != "Approved":
    raise ValueError(
        f"Model package must be Approved before deployment. Current status: {approval_status}"
    )

Model package status: Completed
Approval status: Approved


## Create SageMaker Model from Model Package

Create a SageMaker Model from the approved model package.

The model package contains the registered model artifact and inference container configuration.

In [12]:
create_model_response = sm.create_model(
    ModelName=DEPLOYMENT_MODEL_NAME,
    ExecutionRoleArn=execution_role_arn,
    PrimaryContainer={
        "ModelPackageName": model_package_arn,
    },
)

create_model_response

{'ModelArn': 'arn:aws:sagemaker:eu-central-1:591874026136:model/german-credit-risk-deployment-model-20260628175413',
 'ResponseMetadata': {'RequestId': 'db8364eb-2f8e-484a-9266-74977402fb28',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'db8364eb-2f8e-484a-9266-74977402fb28',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '115',
   'date': 'Sun, 28 Jun 2026 17:56:18 GMT'},
  'RetryAttempts': 0}}

## Deploy Endpoint

Create an endpoint configuration and deploy the model to a SageMaker real-time endpoint.

The endpoint is used for a smoke test and should be deleted after testing if it is not needed anymore.

In [13]:
create_endpoint_config_response = sm.create_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": DEPLOYMENT_MODEL_NAME,
            "InitialInstanceCount": 1,
            "InstanceType": INFERENCE_INSTANCE_TYPE,
            "InitialVariantWeight": 1.0,
        }
    ],
)

create_endpoint_config_response

{'EndpointConfigArn': 'arn:aws:sagemaker:eu-central-1:591874026136:endpoint-config/german-credit-risk-endpoint-config-20260628175413',
 'ResponseMetadata': {'RequestId': '63d3199f-c5f9-401a-a2a0-562cc5b96799',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '63d3199f-c5f9-401a-a2a0-562cc5b96799',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '133',
   'date': 'Sun, 28 Jun 2026 17:56:36 GMT'},
  'RetryAttempts': 0}}

In [14]:
try:
    existing_endpoint = sm.describe_endpoint(
        EndpointName=DEPLOYMENT_ENDPOINT_NAME,
    )

    raise ValueError(
        f"Endpoint already exists: {DEPLOYMENT_ENDPOINT_NAME}. "
        "Delete it first or use a different endpoint name."
    )

except ClientError as error:
    error_code = error.response["Error"]["Code"]

    if error_code != "ValidationException":
        raise

    print("Endpoint does not exist yet. Creating endpoint.")

Endpoint does not exist yet. Creating endpoint.


In [15]:
create_endpoint_response = sm.create_endpoint(
    EndpointName=DEPLOYMENT_ENDPOINT_NAME,
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
)

create_endpoint_response

{'EndpointArn': 'arn:aws:sagemaker:eu-central-1:591874026136:endpoint/german-credit-risk-endpoint',
 'ResponseMetadata': {'RequestId': '61df0774-25e4-4fb1-873f-8938f47442a6',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '61df0774-25e4-4fb1-873f-8938f47442a6',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '98',
   'date': 'Sun, 28 Jun 2026 17:56:56 GMT'},
  'RetryAttempts': 0}}

In [16]:
print(f"Waiting for endpoint to be in service: {DEPLOYMENT_ENDPOINT_NAME}")

waiter = sm.get_waiter("endpoint_in_service")
waiter.wait(
    EndpointName=DEPLOYMENT_ENDPOINT_NAME,
)

print("Endpoint is in service.")

Waiting for endpoint to be in service: german-credit-risk-endpoint


Endpoint is in service.


## Invoke Endpoint

Invoke the deployed endpoint with a small sample from the test dataset.

The payload is sent as CSV without headers, matching the format used during the previous endpoint smoke test.

In [17]:
test_obj = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=TEST_DATA_KEY,
)

test_df = pd.read_csv(test_obj["Body"])

test_df.head()

,duration,credit_amount,installment_commitment,residence_since,age,existing_credits,num_dependents,checking_status_0le_Xlt_200,checking_status_lt_0,checking_status_ge_200,...,job_unemp_unskilled_non_res,job_unskilled_resident,own_telephone_none,own_telephone_yes,foreign_worker_no,foreign_worker_yes,other_payment_plans_bank,other_payment_plans_none,other_payment_plans_stores,class
0,24,1938,4,3,32,1,1,0,1,0,...,0,0,1,0,0,1,0,1,0,0
1,60,14027,4,2,27,1,1,1,0,0,...,0,0,0,1,0,1,0,1,0,0
2,24,2255,4,1,54,1,1,0,0,0,...,0,0,1,0,0,1,0,1,0,1
3,36,15857,2,3,43,1,1,0,1,0,...,0,0,1,0,0,1,0,1,0,1
4,15,1433,4,3,25,2,1,0,1,0,...,0,0,1,0,0,1,0,1,0,1


In [18]:
X_test = test_df.drop(columns=[TARGET_COLUMN])

sample = X_test.head(3)

payload = sample.to_csv(
    header=False,
    index=False,
)

print(payload)

24,1938,4,3,32,1,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,1,0,0,0,1,0,0,1,0,0,1,0,0,1,0,1,0
60,14027,4,2,27,1,1,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,1,0,1,0,0,0,0,1,0,1,0,1,0
24,2255,4,1,54,1,1,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,1,0,1,0,0,0,1,0,0,1,0,0,1,0,0,1,0,1,0



In [20]:
response = sm_runtime.invoke_endpoint(
    EndpointName=DEPLOYMENT_ENDPOINT_NAME,
    ContentType="text/csv",
    Accept="text/csv",
    Body=payload,
)

prediction_response = response["Body"].read().decode("utf-8")

print(prediction_response)

0.4986909329891205
0.301662802696228
0.8488402962684631



## Clean Up Endpoint

Delete the deployed endpoint and endpoint configuration to avoid ongoing costs.

The registered model package remains available in SageMaker Model Registry.

In [22]:
endpoint_description = sm.describe_endpoint(
    EndpointName=DEPLOYMENT_ENDPOINT_NAME,
)

deployed_endpoint_config_name = endpoint_description["EndpointConfigName"]

sm.delete_endpoint(
    EndpointName=DEPLOYMENT_ENDPOINT_NAME,
)

print(f"Deleting endpoint: {DEPLOYMENT_ENDPOINT_NAME}")

sm.get_waiter("endpoint_deleted").wait(
    EndpointName=DEPLOYMENT_ENDPOINT_NAME,
)

print(f"Deleted endpoint: {DEPLOYMENT_ENDPOINT_NAME}")

Deleting endpoint: german-credit-risk-endpoint


Deleted endpoint: german-credit-risk-endpoint


## Result

The approved model package from SageMaker Model Registry was deployed to a SageMaker real-time endpoint.

The endpoint was invoked with sample test records and returned prediction probabilities successfully. After the smoke test, the endpoint and endpoint configuration were deleted to avoid ongoing costs, while the approved model package remains available in the Model Registry.